# Sifu demo: Design stage, risk router, pause and resume

Needs Ollama running with the models in `config.yaml` and `tripplanner-ios` at `workspace.product_repo_path`. Cell 2 runs the five seeded Design-stage tickets. Cells 3 to 5 show the router: a low-risk ticket is approved automatically and a ticket that names a protected file (`.github/`) pauses for you.

The databases in `data/` persist between runs, so tickets and open requests pile up. For a clean run, set `RESET = True` in the next cell, restart the kernel and run it before the others.

In [1]:
from pathlib import Path

RESET = False  # True deletes the demo databases and their checkpoints for a clean run

if RESET:
    # Restart the kernel first so no open connection still holds these files.
    for name in ("design.db", "sifu.db"):
        for path in Path("data").glob(f"{name}*"):  # the .db file plus its -wal and -shm
            path.unlink()
            print(f"removed {path}")

In [2]:
from sifu.demo import run_design_demo

rows = run_design_demo(db_path="data/design.db")
rows

ok clean         expected=spec      got=spec      status=spec         events=15
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/1/spec.md
   reviewer:scope       0 findings
   reviewer:ambiguity   0 findings
   reviewer:policy      1 findings
ok ambiguous     expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/2/questions.md
   reviewer:scope       3 findings
   reviewer:policy      1 findings
   reviewer:ambiguity   3 findings
ok out_of_scope  expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/3/questions.md
   reviewer:policy      3 findings
   reviewer:scope       1 findings
   reviewer:ambiguity   6 findings
ok contradiction expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/4/questions.md
   reviewer:scope       2 findings
   

[{'key': 'clean',
  'ticket_id': 1,
  'expected': 'spec',
  'outcome': 'spec',
  'status': 'spec',
  'events': 15,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/1/spec.md'},
 {'key': 'ambiguous',
  'ticket_id': 2,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/2/questions.md'},
 {'key': 'out_of_scope',
  'ticket_id': 3,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/3/questions.md'},
 {'key': 'contradiction',
  'ticket_id': 4,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/4/questions.md'},
 {'key': 'injection',
  'ticket_id': 5,
  'expected': 'questions',
  'outcome': 'qu

In [3]:
from sifu.demo import open_demo, run_router_demo

deps, project_id = open_demo(db_path="data/sifu.db")
ids = run_router_demo(deps, project_id)

ticket 1 (clean): continued
ticket 2 (protected_path): PAUSED for a human
decision 1 ticket 1 accept_spec: floor=0 final=1 rules=["assessor_disagreement"] -> auto_with_veto
decision 2 ticket 2 accept_spec: floor=2 final=2 rules=["touches_protected_path"] -> pending_human
OPEN request 1 (ticket 2): ["approve", "close"]


The second ticket is paused in the SQLite checkpoint. Look at the open request. If it asks you to approve, answer `approve` or `close`, or any other text as feedback. If it asks you to answer questions, answer with text (it goes back to the reviewers as a clarification) or `close`.

In [4]:
from sifu import db
from sifu.demo import answer_request

# Answer the request this run's protected ticket opened. A reused database file can hold
# older open requests too, so pick it by ticket rather than by a fixed id.
[request] = [r for r in db.open_requests(deps.conn) if r["ticket_id"] == ids["protected_path"]]
state = answer_request(deps, request["id"], "approve")

ticket 2: resumed, status=spec
decision 1 ticket 1 accept_spec: floor=0 final=1 rules=["assessor_disagreement"] -> auto_with_veto
decision 2 ticket 2 accept_spec: floor=2 final=2 rules=["touches_protected_path"] -> human_approved


## Evaluation results

These cells read only the results files in `data/artifacts/`, with no model or GitHub calls.
That folder is git-ignored, so they run where the evaluation ran. To make new files, run `scripts/run_eval.py` (Design stage) and
`scripts/golden_critics.py` (critics).

In [ ]:
from IPython.display import Markdown

from sifu.report import (
    critic_table,
    design_table,
    latest,
    load,
    tokens_by_case,
    wrong_tickets,
)

v1, v2 = load(latest("eval", "main")), load(latest("eval", "v2"))
Markdown(design_table({"v1": v1, "v2": v2}))

### Design stage: swarm against the single-prompt baseline

23 tickets, 3 repeats each. Both systems get the same product context and the same deterministic
checks and risk floor. How to read it:

- **Detection is at the ceiling.** Both systems catch every seeded defect that needs a model, so
  this set cannot separate them on detection. The swarm's edge is elsewhere, or would need harder
  tickets.
- **v2 differs from v1 in five ways at once:** the rewritten `protected_ci` ticket, the glossary
  rewrite before the models see the ticket, the `requirement_link` policy prompt, the
  `qwen3.8:27b-mlx` models and the 8,192-token output cap. The drop in false positives and
  over-escalation cannot be put down to one of them.
- **Baseline glossary detection reads 0% in v2.** The baseline only sees the rewritten text, so it
  never reports the avoided word. The swarm runs its glossary check on the original text. The
  outcome is right for both (spec, auto); this is a scoring artifact of the rewrite, not a miss.
- **The swarm costs about 3.7 times the tokens and 2.5 times the time** of the baseline, for the
  same accuracy on this set.
- v1 ran before results files were stamped with a commit.

In [ ]:
Markdown(wrong_tickets(v2))

`protected_ci` is blocked on every repeat by both systems. The reason the models give is right:
`swift test --enable-code-coverage` does not write a file called `coverage.json` (SwiftPM writes
`.build/debug/codecov/<package>.json`). The ticket, not the model, is wrong, so this "false
positive" is a defect in the golden ticket. Fix the ticket before the next run rather than
relabel it.

In [ ]:
alt_v2 = load(latest("eval", "alt-v2"))
Markdown(design_table({"v2 qwen": v2, "v2 gemma": alt_v2}, systems=["swarm"]))

### Spec reviewer: `qwen3.8:27b-mlx` against `gemma4:31b-mlx`

Swarm only, the reviewer model swapped with `--set models.spec_reviewer=gemma4:31b-mlx`.
Gemma is worse on every count that matters: false positives 50% against 12%, over-escalation 56% against 0%, right outcome 83% against 96%, and only 87% of tickets get the same outcome on every repeat. It blocks clean tickets (`clean_day_total` on all three repeats, `clean_stop_order`, `clean_min_duration`) with `missing_edge_case` findings the spec answers elsewhere, as it did in the v1 alt run. Keep qwen as the spec reviewer. Its tokens per ticket are a little lower (9,150 against 10,589).

In [ ]:
main = load("data/artifacts/golden/critics-20261114-075911.json")
heldout = load("data/artifacts/golden/critics-20261114-000941.json")
heldout_checks = load("data/artifacts/golden/critics-20261114-072557.json")  # --checks-only
Markdown(critic_table({"main (10 cases)": main, "held-out (4 cases)": heldout},
                      {"held-out (4 cases)": heldout_checks}))

### Critics on real diffs: what they find beyond the deterministic checks

Each golden case is a patch to the product repo with seeded flaws, or a clean twin. Sifu's own
verify steps (build with warnings as errors, SwiftLint strict, `swift test`) run on each patch
first.

- **The critics find 20 of 20 seeded flaws; the checks catch 1** (req6's `try!`, through SwiftLint
  `force_try`). So 19 flaws are found only by the critics.
- **False positives: 0**, on 7 clean diffs that pass verify. This is the critics' false-positive
  rate on this set; with 7 clean cases it is an upper bound worth stating as "0 of 7", not "0%".
- **Exact line against the 2-line window:** 15 of 16 main-set flaws are cited on the line the
  label quotes. The one miss cites the `throws` signature two lines above.
- **The main set was used to tune the critic rubrics.** The held-out cases were written after the
  first run and never used for tuning, but they ran once before the prompts were frozen, with the
  blocker checker on `qwen3.8:27b-mlx` where the tuning run used `qwen3.8:27b`.
- Detection is at the ceiling here too. Harder or subtler flaws are the next step, not more of the
  same kind.
- The critics are not free: about 6–27k tokens per diff (below).

In [ ]:
Markdown(tokens_by_case(main))